
What about taking proofs baout T and a mapping function

proofs = {
    p : p2
}
decls : {

}

def tr

Hmm. But how do I specify transporting quantifiers?

smt.substitute or substitute_funs requires transports to be sort preserving


Kind of transporting sorts and transporting Predicates might be interrelated.

It would be nice to point out the stuff one wants transported, not just transport every Int.
But that's why you should point it out in the schematic as T.

The identity transportation just replays the proof.

https://arxiv.org/abs/2303.05244 Transport via Partial Galois Connections and Equivalences
Lifting package

https://inria.hal.science/hal-05192017v1/document Trocq: Proof Transfer for Free, Beyond Equivalence and
Univalence

The sorts of games systems like Rocq may have to play for wacky stuff

CoqEAL
Pumpkin Patch

contraviariant and covari
System F ought to be an interface to something just as much as ACL

ACL2 functional instantiation and encapsulation

Turn internalized stuff into python lambdas. Can parametrize 
def encapsulate([T,f,...], pf : )
def encapulate([T,f,..], body):





In [ ]:
# relational model?
def RelForAll(vs, body):
    smt.ForAll([R, v1, v2], R(v,v1), )

# Transport a function into a function on pairs?
def RelLambda(vs, body, substs, sorts):
    smt.Const( Tuple()
    transport(body, {T : kd.Tuple(v.sort(), sorts.get(v.sort(), v.sort())})
    smt.Lambda([tupv], kd.tuple_(body, transport(body, sorts)))



In [ ]:
def QuotForAll(vs, body):
    for v,eq in vs:
    
    smt.ForAll(vs1 + vs2, smt.) # well if it's true forall, we're good


def QuotExists(vs, body):
    smt.And(smt.Exists([vs], body), smt.ForAll(vs1 + vs2, body == smt.substitute(body, zip(vs1,vs2))))

def WFLambda(f, eq):
    smt.ForAll([vs], eq(v1,v2), eq(f(v1), f(v2)))

Where do isomorphisms or galois functions pop in?

I guess if I want to transport just a subterm I'd need to : A -> B
If I want to transport a context, I need from : B -> A

pathmap

https://link.springer.com/chapter/10.1007/978-3-319-03545-1_9 Lifting and Transfer: A Modular Design for Quotients in Isabelle/HOL


Maybe casts _only_ apply to binders.

forall x : int, body[x]  ---> forall y : nat, body[cast(y)]

In [ ]:
class Iso():
    to : smt.FuncDeclRef
    from_ : smt.FuncDeclRef
    to_from : kd.Proof
    from_to : kd.Proof

In [ ]:
def transport(e, expected_sort=None, casts : dict[tuple[smt.SortRef, smt.SortRef], smt.FuncDeclRef]): # isos?
    if e.sort() == expected_sort:
        return e
    else:
        casts[(e.sort(), expected_sort)](e)
    if smt.is_const(e):
        return e
    elif smt.is_app(e):
        decl, children = e.decl(), e.children()
        children = [transport(c, casts) for c in children]
        if (decl.range(), decl.domain()) in casts:
            return casts[(decl.range(), decl.domain())](*children)
        else:
            return decl(*children)
    else:
        raise ValueError("unexpected term", e)

In [ ]:
from kdrag.all import *

def transport_decl(decl : kd.Decl, decls={}, consts={}, sorts={}):
    if decl in decls:
        return decls[decl]
    else:
        doms = [decl.domain(i) for i in range(decl.arity())]
        doms = [sorts.get(d, d) for d in doms]
        ran = sorts.get(decl.range(), decl.range())
        return smt.Function(decl.name(), *doms, ran)
        
def transport_sort(s : kd.Sort, decls={}, consts={}, sorts={}):
    if s in sorts:
        return sorts[s]
    else:
        return s

def transport_consts(c : smt.ExprRef, decls={}, consts={}, sorts={}):
    if c in consts:
        return consts[c]
    elif c.sort() in sorts:
        return smt.Const(c.decl().name(), sorts[c.sort()])
    else:
        return c

def transport(e : smt.ExprRef, decls={}, consts={}, sorts={}) -> smt.ExprRef:
    if smt.is_const(e):
        return transport_consts(e, decls, consts, sorts)
    elif smt.is_app(e):
        decl, children = e.decl(), e.children()
        children = [transport(c, decls, consts, sorts) for c in children]
        if decl in decls:
            return decls[decl](*children)
        else:
            return decl(*children)
    else:
        raise ValueError("unexpected term", e)
x = smt.Int("x")
y = smt.Bool("y")

transport(x+x+x*x, decls={(x + x).decl() : (y | y).decl(), (x * x).decl() : (y & y).decl()}, consts={x : y})

# axioms do not change type. Stay Bool

def transport_pf(pf : kd.Proof, decls={}, consts={}, axioms={}):
    thm = smt.substitute(transport(pf.thm, decls, consts), *[(p.thm,p2.thm) for p1,p2 in axioms.items())
    if pf in axioms:
        return axioms[pf]
    match pf.reason:
        case ["prove", *by]:
            return kd.prove(thm, by=[transport_pf(p, decls, consts, axioms) for p in by])
        case ["axiom"]:
            raise ValueError("axiom not in axioms", pf)
        case ["herbrand", ]:
            # yikes.  Maybe peek to add these in to consts?
            # but also 
        case ["modus", pf1, pf2]:
            pf1 = transport_pf(pf1, decls, consts, axioms)
            pf2 = transport_pf(pf2, decls, consts, axioms)
            return kd.modus_ponens(pf1, pf2)
        


Or(Or(y, y), And(y, y))

In [ ]:
from kdrag.all import *
def transport(e : smt.ExprRef, subst, sorts) -> smt.ExprRef:
    if smt.is_app(e):
        decl, children = e.decl(), e.children()
        children = [transport(c, subst) for c in children]
        if decl in subst:
            subst[decl](*children)
        else:
            decl(*children)
    elif isinstance(e, QuantifierRef):
        vs, body = open_binder_unhygienic(e)
        body1 = transport(body, subst, sorts)
        vs1 = []
        for v in vs:
            if v.sort() in sorts:
                vs1.append(smt.Const(v.name(), v.sort()))
            else:
                vs1.append(v)
        if e.is_forall():
            return smt.ForAll(vs1, body1)
        elifve.is_exists():
            


# pfs may contain axioms.
# but also maybe want to note induction principles, others?

def transport_pf(p : Proof, subst : dict[smt.FuncDeclRef, smt.FuncDeclRef], pfs : dict[Id, kd.Proof]) 
    id = p.thm.get_id()
    if id in pfs:
        return pfs[id]
    else:


{
groupT.decl() : groupInt.decl(), 
mulT.decl() : addInt.decl(),
}

{
    groupT.defn : groupInt.defn,

}

